In [ ]:
"""
====================================================
ERA5 Visualization Script
====================================================
"""

In [ ]:
#######################
#DIRECTORIES

In [ ]:
#SETTING UP DIRECTORIES
mainDirectory = '/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/'
workingDirectory="/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/DataAnalysis/InputData_DataAnalysis/"
print(workingDirectory)
outputDirectory=workingDirectory+"OUTPUT/"
dataDirectory=mainDirectory+"DownloadData/DATA/ERA5_Data/"

In [ ]:
#######################
#LIBRARIES, FUNCTIONS, and CLASSES

In [ ]:
#IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Libraries/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
#IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/Classes/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Classes_1",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [16]:
###########################
#FUNCTIONS

In [22]:
def RunCalculations(var_data, units, variable, calculation, numerics):
    calculation_results = {}

    arr   = var_data
    units = units

    tz, _ = Ultimate_AreaAverage(var_data, dims=('t','z','y','x'), dim_names=('t','z'), mode='keep')
    t, _  = Ultimate_AreaAverage(tz,  dims=('t','z'),        dim_names=('t',),   mode='keep')

    three_hours = 3 * numerics.hour_index
    tz_3h   = calculation.block_vertical_profiles_2D(tz,  block=three_hours)
    tzyx_3h = calculation.block_vertical_profiles_4D(arr, block=three_hours)

    calculation_results[variable] = {
        "units": units,
        "tz": tz,          # (t,z)
        "t": t,            # (t,)
        "tz_3h": tz_3h,    # (nblocks, z)
        "tzyx_3h": tzyx_3h # (nblocks, z, y, x)
    }

    return calculation_results
    
def RunPlots(calculation_results, date_string, outputFile, plotting):
    for name, result in calculation_results.items():
        common_args = {
            "var_name": name,
            "var_units": result["units"],
            "date_string": date_string,
            "date_folder":  date_folder,
            "outputFile": outputFile
        }

        plotting.TZContourPlot(var_data=result['tz'], **common_args)
        plotting.TimeSeries(var_data=result['t'], numerics=numerics, **common_args)
        plotting.MultiAverage_VerticalProfiles(var_data=result['tz_3h'], **common_args)
        plotting.MultiAverage_HorizontalFields(var_data=result['tzyx_3h'], plev=1000, **common_args)

def FixVariableName(variable,var_name,var_data):
    if variable == 'divergence':
        variable = 'convergence'
        var_name = variable
        var_data*= 1
    return variable,var_name,var_data

In [23]:
###########################
#LOADING DATA

In [24]:
###########################
#DATE ONE (BORING CASE)

In [25]:
#date information
date_string = "06-08 - 06-10 (2022)"
date_folder = strings.DateString(date_string)
#adding date to output folder
subdir = os.path.join(outputDirectory, date_folder)
os.makedirs(subdir, exist_ok=True)

In [29]:
#load in ERA5 data
variables = {
    "u_component_of_wind": {"unit": r"$m\ s^{-1}$"},
    # "v_component_of_wind": {"unit": r"$m\ s^{-1}$"},
    # "vertical_velocity": {"unit": r"$Pa\ s^{-1}$"},
    # "divergence": {"unit": r"$s^{-1}$"},
    # "vorticity": {"unit": r"$s^{-1}$"},
    # "temperature": {"unit": r"$K$"},
    # "specific_humidity": {"unit": r"$kg\ kg^{-1}$"},
    # "specific_cloud_liquid_water_content": {"unit": r"$kg\ kg^{-1}$"},
    # "specific_cloud_ice_water_content": {"unit": r"$kg\ kg^{-1}$"},
    # "specific_rain_water_content": {"unit": r"$kg\ kg^{-1}$"},
    # "relative_humidity": {"unit": r"$\%$"},
    # "cloud_cover": {"unit": r"$1$"},      # fraction 0–1
    # "geopotential": {"unit": r"$m^{2}\ s^{-2}$"}
}

loadDirectorys = [
    os.path.join(dataDirectory, date_folder, f"{var}_ERA5_{date_folder}.nc")
    for var in variables.keys()
]
loadDirectorys

['/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/DownloadData/DATA/ERA5_Data/06-08_-_06-10_2022/u_component_of_wind_ERA5_06-08_-_06-10_2022.nc']

In [27]:
###########################
#RUNNING

In [38]:
#RUNNING CALCULATIONS
for count, (loadDirectory, (variable, units)) in enumerate(tqdm(zip(loadDirectorys, variables.items()), total=len(loadDirectorys), desc="Running Calculations"), start=1):
    #print
    print(f"Plotting {len(variables)} Variables",'\n')
    print(f"{count}. {variable} ({units['unit']}) → {loadDirectory}")
    
    #loading the variable
    ncFile=xr.open_dataset(loadDirectory)
    var_name = [v for v in list(ncFile.data_vars) if v not in ["number", "expver"]][0]
    var_data=ncFile[var_name].data
    [variable,var_name,var_data] = FixVariableName(variable,var_name,var_data)
    numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2)
    print(variable+":\n","\t(Nt, Np, Nlat, Nlon) = ",(numerics.Nt,numerics.Np,numerics.Nlat,numerics.Nlon),"\n")

    #making output filename
    outputFile = os.path.join(outputDirectory, date_folder, variable) #variable also can be var_name
    print('\n',var_name,'***')
    os.makedirs(outputFile, exist_ok=True)

    #doing calculations
    calculation_results=RunCalculations(var_data, "("+units['unit']+")", variable, calculation, numerics)

    #plotting
    RunPlots(calculation_results, date_string, outputFile, plotting)

Running Calculations:   0%|          | 0/1 [00:00<?, ?it/s]

Plotting 1 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/DownloadData/DATA/ERA5_Data/06-08_-_06-10_2022/u_component_of_wind_ERA5_06-08_-_06-10_2022.nc
u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 5, 20, 23) 


 u ***


Running Calculations: 100%|██████████| 1/1 [00:03<00:00,  3.96s/it]


In [ ]:
#*#* Need to fix some things in the figures #*#*
#1. TimeSeries: xtick labels, also lines are misaligned
#2. VerticalProfiles: zero axvline

#OVERALL: 
#1. need to use different colorbars for different variables
#2. colorbar ticks

In [37]:
#PLOTTING FUNCTIONS

#1. (T,Z) Contour Plot
class Plotting:
    def __init__(self):
        pass

    #1. (T,Z) Contour Plot
    def TZContourPlot(self, var_data, var_name, var_units, date_string, date_folder, outputFile):
        #setting up labels
        pc = np.linspace(1000,1,var_data.shape[1])     # vertical levels
        times = np.arange(0, var_data.shape[0]*3, 3)  # [0,3,6,9,...]
        labels = [f"{t}-{t+3-1} h" for t in times]
    
        #setting up plot figure
        fig = plt.figure(figsize=(10,4))
        gs = gridspec.GridSpec(1, 1, figure=fig)
    
        #plotting
        ax = fig.add_subplot(gs[0, 0])
        cf = ax.contourf(times, pc, var_data.T, cmap="RdBu_r")
        cbar = fig.colorbar(cf, ax=ax)
        cbar.set_label(f"{var_name} {var_units}")
    
        #inverting yaxis
        ax.set_ylim(1000, 1) 
    
        #labels
        ax.set_xlabel('Time (hrs)')
        ax.set_ylabel('p ' + r'$(hrs)$')
        # ax.set_ylabel(f"{var_name} {var_units}")
        ax.set_title(f"TZ Contour of {var_name} {var_units} \nERA5 Data on {date_string}")

        #saving plot
        fig.savefig(os.path.join(outputFile, f"{var_name}_TZContour_{date_folder}.jpg"))
        plt.close(fig)   # ensures it won’t show up in Jupyter
        

    #2. TIME SERIES    
    def TimeSeries(self, var_data, var_name, var_units, date_string, date_folder, outputFile, numerics):
        #setting up plot figure
        fig = plt.figure(figsize=(10,4))
        gs = gridspec.GridSpec(1, 1, figure=fig)
    
        #plotting
        ax = fig.add_subplot(gs[0, 0])
        ax.plot(numerics.time/3600, var_data, lw=1.5)
    
        #labels
        ax.set_xlabel('Time (hrs)')
        ax.set_ylabel(f"{var_name} {var_units}")
        ax.set_title(f"Time-series of {var_name} {var_units} \nERA5 Data on {date_string}")
    
        # Set xticks every 3 hours
        max_hours = numerics.time[-1]/3600
        ax.set_xticks(np.arange(0, max_hours+1, 3))
    
        #vertical lines
        ndays=3; vline_xinds = np.insert(np.arange(24, 24*ndays+1, 24) - 1, 0, 0) # e.g. every 24 hrs
        for x in vline_xinds:
            ax.axvline(x, color='k', linestyle='--', alpha=0.7)
        ax.axvline(24-18-1,color='blue', linestyle='--', alpha=0.7, label='model start-time')
        
        #other
        fig.tight_layout()
        fig.legend()

        #saving plot
        fig.savefig(os.path.join(outputFile, f"{var_name}_TimeSeries_{date_folder}.jpg"))
        plt.close(fig)   # ensures it won’t show up in Jupyter
    
    #3. VERTICAL PROFILES
    def MultiAverage_VerticalProfiles(self, var_data,var_name,var_units,date_string, date_folder, outputFile):
        #setting up labels
        pc = np.linspace(1000,1,var_data.shape[1])     # vertical levels
        times = np.arange(0, var_data.shape[0]*3, 3)  # [0,3,6,9,...]
        labels = [f"{t}-{t+3-1} h" for t in times]
        
        #setting up number of plots
        nplots = var_data.shape[0]
        cols = 8                                      # 8 profiles per row
        rows = int(np.ceil(nplots / cols))            # number of rows needed
        
        #setting up plot figure
        fig = plt.figure(figsize=(2*cols, 3*rows), constrained_layout=True)   # scale fig size to rows/cols
        gs = gridspec.GridSpec(rows, cols, figure=fig, wspace=0.1)
        
        axes = []
        for i, t in enumerate(times):
            r = i // cols   # row index
            c = i % cols    # column index
            ax = fig.add_subplot(gs[r, c])
            ax.plot(var_data[i, :], pc)
    
            #inverting yaxis
            ax.invert_yaxis()
            
            #labels
            ax.set_title(labels[i], fontsize=12)
            ax.set_xlabel(f"{var_name} {var_units}",fontsize=9)
            if c == 0:
                ax.set_ylabel("p (hPa)")
            else:
                ax.set_yticklabels([])  # hide y tick labels except first col
            # axes.append(ax)
        
        #fixing xlims
        axes = fig.get_axes()
        MatchAxisLimits(axes, dim='x')
        
        fig.suptitle(f"Vertical Profiles of {var_name} {var_units} \nERA5 Data on {date_string}")

        #saving plot
        fig.savefig(os.path.join(outputFile, f"{var_name}_VerticalProfiles_{date_folder}.jpg"))
        plt.close(fig)   # ensures it won’t show up in Jupyter

    #4. Horizontal Fields
    def MultiAverage_HorizontalFields(self, var_data, var_name, var_units, date_string, date_folder, plev, outputFile, cmap="RdBu_r"):
        """
        Plot horizontal contour maps at a given pressure level for each block in var_data,
        with a single consistent colorbar.
        """
        nblocks, Nz, Ny, Nx = var_data.shape
        pc = np.linspace(1000, 1, Nz)       # pressure coords
        yc = np.arange(0,Ny,1)
        xc = np.arange(0,Nx,1)
        pind = np.argmin(np.abs(pc - plev)) # nearest index
        times = np.arange(0, nblocks*3, 3)  # hours (assuming 3h blocks)
        labels = [f"{t}-{t+3-1} h" for t in times]
    
        # layout
        cols = 8
        rows = int(np.ceil(nblocks / cols))
    
        # global color limits
        vmin = np.min(var_data[:, pind, :, :])
        vmax = np.max(var_data[:, pind, :, :])
    
        fig = plt.figure(figsize=(2.5*cols, 2.5*rows), constrained_layout=True)
        gs = gridspec.GridSpec(rows, cols, figure=fig, wspace=0.1)
    
        mappable = None  # will store the last contourf object
    
        for i in range(nblocks):
            r = i // cols
            c = i % cols
            ax = fig.add_subplot(gs[r, c])
    
            # horizontal slice at given z index
            field = var_data[i, pind, :, :]   # (Ny, Nx)
            cf = ax.contourf(xc,yc,field, cmap=cmap, vmin=vmin, vmax=vmax)
    
            ax.set_title(labels[i], fontsize=12)
            mappable = cf
    
        # one shared colorbar
        cbar = fig.colorbar(mappable, ax=fig.get_axes(), orientation="vertical", shrink=0.6)
        cbar.set_label(f"{var_name} {var_units}")
    
        fig.suptitle(f"Horizontal Fields of {var_name} {var_units} at p={plev} hPa \nERA5 Data on {date_string}")

        #saving plot
        fig.savefig(os.path.join(outputFile, f"{var_name}_HorizontalFields_{date_folder}.jpg"))
        plt.close(fig)   # ensures it won’t show up in Jupyter

plotting = Plotting()

array([ 0, 23, 47, 71])